# LLM Fine-tuning: LoRA & QLoRA Tutorial

This notebook demonstrates:
- Loading Qwen3 with 4-bit quantization (QLoRA)
- Applying LoRA adapters via PEFT
- Running supervised fine-tuning (SFT)
- Inference with the trained adapter

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments

MODEL_NAME = "Qwen/Qwen3-8B"

## 1. Load Tokenizer and Quantized Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

## 2. Attach LoRA Adapter

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. Prepare Dataset

Use the `build_sft_dataset` utility from `src/data_utils.py`.

In [ ]:
import sys
sys.path.append("../src")
from data_utils import build_sft_dataset

dataset = build_sft_dataset(
    "../data/alpaca_zh.jsonl",
    template="alpaca",
    tokenizer=tokenizer,
    max_seq_length=512
)

## 4. Train

In [ ]:
training_args = TrainingArguments(
    output_dir="../outputs/qwen3_lora_nb",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    bf16=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args,
)
trainer.train()

## 5. Save and Inference

In [ ]:
model.save_pretrained("../outputs/qwen3_lora_nb/final_adapter")
tokenizer.save_pretrained("../outputs/qwen3_lora_nb/final_adapter")

inputs = tokenizer("### Instruction:\nIntroduce LoRA\n\n### Response:\n", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=128)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))